In [1]:
import numpy as np

import os 
import sys
sys.path.append("..//utils/")
sys.path.append("..//anatomy/")
import color_utils, make_data_dict
import get_probe_coords
import format_waveform_data, waveform_analysis, waveform_plots
import matplotlib.pyplot as plt

In [2]:
''' Set file paths '''
root_dir = "Z:/Isabel/data/hpc_implants/"
data_file = f"{root_dir}stim_session_data.npy"
session_info_file = f"{root_dir}good_sessions.xlsx"
save_figs = f"../figures/antidromic_hpc_to_lhy/"

In [4]:
''' Load the dictionary of waveform data for all good stim sessions '''
bird_ids = []
data_dict = np.load(data_file, allow_pickle=True).item()
for bird in data_dict.keys():
    bird_ids.append(bird)
print(f'current birds with saved data: {bird_ids}')

# check for stim data
stim_sessions = []
collision_sessions = []
for bird in bird_ids:
    session_list = data_dict[bird]['all_sessions']
    for session_id in session_list:
        # get the list of stim sessions
        if 'worm_ch_idx' in data_dict[bird][session_id].keys():
            stim_sessions.append(f'{bird}_{session_id}')

        # get the list of collision sessions
        if 'proj_cell_IDs' in data_dict[bird][session_id].keys():
            collision_sessions.append(f'{bird}_{session_id}')

current birds with saved data: ['LIM63', 'RBY94', 'AMB154', 'SLV132', 'IND67', 'LMN146']


In [8]:
''' Plot all collision cells in brain space '''
all_proj_idx = np.asarray([])
all_cell_pos = []
for bird in bird_ids:
    bird_dir = f"{root_dir}{bird}/"
    session_list = data_dict[bird]['all_sessions']
    for session_id in session_list:
        if f'{bird}_{session_id}' in collision_sessions:
            # get the cell IDs
            session_dir = f'{bird_dir}/{bird}_{session_id}/'
            for folder in sorted(os.listdir(session_dir)):
                if f'{bird}_{session_id}' in folder:
                    ephys_id = folder[-13:]
                    for file in sorted(os.listdir(f"{session_dir}{bird}_{ephys_id}")):
                        if 'kilosort4' in file:
                            data_dict[bird][session_id]['ephys_id'] = ephys_id
                            ks_dir = f"{bird}_{ephys_id}/{file}/"
                            ephys_dir = f"{session_dir}{bird}_{ephys_id}/raw_ephys_output/"

                            # load and format the waveform struct
                            waveform_struct = format_waveform_data.load_wf_data(session_dir, ks_dir=ks_dir)
                            wf_ids = waveform_struct['goodIDs']
                            
            # get the cell positions
            cell_pos = data_dict[bird][session_id]['cell_pos']
            if len(all_cell_pos) == 0:
                all_cell_pos = cell_pos
            else:
                all_cell_pos = np.row_stack([all_cell_pos, cell_pos])

            # get the indices for the collision-verified cells and convert to bool
            proj_idx = data_dict[bird][session_id]['proj_cell_idx']

            print(f'{bird}_{session_id} putative projection cells: {wf_ids[proj_idx]}\n')

Z:/Isabel/data/hpc_implants/LIM63//LIM63_240610/LIM63_240610_131820/kilosort4_blanked/waveformStruct.mat
LIM63_240610 putative projection cells: [ 56  57 131 148 176 246 278 309]

Z:/Isabel/data/hpc_implants/AMB154//AMB154_241119/AMB154_241119_110619/kilosort4_blanked/waveformStruct.mat
AMB154_241119 putative projection cells: [114 130]

Z:/Isabel/data/hpc_implants/AMB154//AMB154_241122/AMB154_241122_110236/kilosort4_blanked/waveformStruct.mat
AMB154_241122 putative projection cells: [385]

Z:/Isabel/data/hpc_implants/AMB154//AMB154_241127/AMB154_241127_110627/kilosort4_blanked/waveformStruct.mat
AMB154_241127 putative projection cells: [368]

Z:/Isabel/data/hpc_implants/IND67//IND67_250929/IND67_250929_103756/kilosort4_blanked/waveformStruct.mat
IND67_250929 putative projection cells: [428 485 539]

Z:/Isabel/data/hpc_implants/IND67//IND67_251001/IND67_251001_104508/kilosort4_blanked/waveformStruct.mat
IND67_251001 putative projection cells: [277 557 565]

Z:/Isabel/data/hpc_implants/

In [31]:
data_dict = np.load(data_file, allow_pickle=True).item()

In [34]:
shuff_file = f"{root_dir}stim_session_shuffles.npy"

In [36]:
''' List of behavior sessions '''
all_behavior_sessions = []
for i, bird in enumerate(bird_ids):
    behavior_sessions = []
    for session_id in data_dict[bird]['all_sessions']:
        preprocessed_data = data_dict[bird][session_id]['preprocessed_data']
        if ('behavior' in preprocessed_data) & ('ephys' in preprocessed_data):
            behavior_sessions.append(session_id)
    all_behavior_sessions.append(behavior_sessions)

shuff_dict = {}
for bird in bird_ids:
    shuff_dict[bird] = {}
    bird_idx = bird_ids.index(bird)
    behavior_sessions = all_behavior_sessions[bird_idx]
    for session_id in behavior_sessions:
        shuff_dict[bird][session_id] = {}
        shuff_avg_cache = data_dict[bird][session_id]['barcode_dict']['shuff_avg_cache'].copy()
        shuff_dict[bird][session_id]['shuff_avg_cache'] = shuff_avg_cache
        data_dict[bird][session_id]['barcode_dict']['shuff_avg_cache'] = []

np.save(data_file, data_dict)
np.save(shuff_file, shuff_dict)

In [33]:
shuff_avg_cache.shape

(78, 87, 1000)

In [24]:
np.save(data_file, data_dict)

TypeError: only integer scalar arrays can be converted to a scalar index

In [30]:
np.linspace(0, 100, 26)

array([  0.,   4.,   8.,  12.,  16.,  20.,  24.,  28.,  32.,  36.,  40.,
        44.,  48.,  52.,  56.,  60.,  64.,  68.,  72.,  76.,  80.,  84.,
        88.,  92.,  96., 100.])